In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [4]:
df = pd.read_csv(r'C:\Users\PL\Documents\ML data\customer_churn_dataset-training-master.csv')

X = df.drop(['Churn','CustomerID'],axis=1)
y = df['Churn']

X['Age'] = X['Age'].fillna(X['Age'].median())
X['Gender'] = X['Gender'].fillna(X['Gender'].mode()[0])
X['Gender'] = X['Gender'].map({'Male':1,'Female':0})
X['Tenure'] = X['Tenure'].fillna(X['Tenure'].median())
X['Usage Frequency'] = X['Usage Frequency'].fillna(X['Usage Frequency'].median())
X['Support Calls'] = X['Support Calls'].fillna(X['Support Calls'].median())
X['Payment Delay'] = X['Payment Delay'].fillna(X['Payment Delay'].median())
X['Subscription Type'] = X['Subscription Type'].fillna(X['Subscription Type'].mode()[0])
X['Contract Length'] = X['Contract Length'].fillna(X['Contract Length'].mode()[0])
X['Total Spend'] = X['Total Spend'].fillna(X['Total Spend'].median())
X['Last Interaction'] = X['Last Interaction'].fillna(X['Last Interaction'].median())

mask = y.notnull()
X = X[mask]
y = y[mask]

X = pd.get_dummies(
    X,columns = ['Subscription Type','Contract Length'],
    drop_first=True)

In [5]:
X.head()

,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Total Spend,Last Interaction,Subscription Type_Premium,Subscription Type_Standard,Contract Length_Monthly,Contract Length_Quarterly
0,30.0,0,39.0,14.0,5.0,18.0,932.0,17.0,False,True,False,False
1,65.0,0,49.0,1.0,10.0,8.0,557.0,6.0,False,False,True,False
2,55.0,0,14.0,4.0,6.0,18.0,185.0,3.0,False,False,False,True
3,58.0,1,38.0,21.0,7.0,7.0,396.0,29.0,False,True,True,False
4,23.0,1,32.0,20.0,5.0,8.0,617.0,20.0,False,False,True,False


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440833 entries, 0 to 440832
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   CustomerID         440832 non-null  float64
 1   Age                440832 non-null  float64
 2   Gender             440832 non-null  object 
 3   Tenure             440832 non-null  float64
 4   Usage Frequency    440832 non-null  float64
 5   Support Calls      440832 non-null  float64
 6   Payment Delay      440832 non-null  float64
 7   Subscription Type  440832 non-null  object 
 8   Contract Length    440832 non-null  object 
 9   Total Spend        440832 non-null  float64
 10  Last Interaction   440832 non-null  float64
 11  Churn              440832 non-null  float64
dtypes: float64(9), object(3)
memory usage: 40.4+ MB


In [7]:
X.isna().sum()
y.isna().sum()

print(X.isna().sum())
print(y.isna().sum())

Age                           0
Gender                        0
Tenure                        0
Usage Frequency               0
Support Calls                 0
Payment Delay                 0
Total Spend                   0
Last Interaction              0
Subscription Type_Premium     0
Subscription Type_Standard    0
Contract Length_Monthly       0
Contract Length_Quarterly     0
dtype: int64
0


In [8]:
X_train, X_test, y_train, y_test = train_test_split ( 
    X, y, test_size = 0.25, random_state = 42, stratify = y )

In [9]:
C_values = [0.01, 0.1, 1, 10, 100]

for C in C_values:
    model = LogisticRegression(C=C, max_iter=5000)
    model.fit(X_train,y_train)
    train_score = model.score(X_train, y_train)
    test_score = model.score(X_test, y_test)
    print(f"C={C} | Train = {train_score:.3f} | Test = {test_score:.3f}")

C=0.01 | Train = 0.895 | Test = 0.894
C=0.1 | Train = 0.896 | Test = 0.894
C=1 | Train = 0.896 | Test = 0.894
C=10 | Train = 0.896 | Test = 0.894
C=100 | Train = 0.896 | Test = 0.894


## Hyperparameter Tuning

C = 0.01 was selected because all tested C values produced the same test score of 0.894. When test scores are tied, the lower C value is preferred because it keeps the Logistic Regression model simpler and reduces the risk of overfitting.

| C Value | Train Score | Test Score |
|---:|---:|---:|
| 0.01 | 0.895 | 0.894 |
| 0.1 | 0.896 | 0.894 |
| 1 | 0.896 | 0.894 |
| 10 | 0.896 | 0.894 |
| 100 | 0.896 | 0.894 |

In [10]:
model = LogisticRegression(C=0.01,max_iter=5000)
model.fit(X_train, y_train)

pred = model.predict(X_test)
probs = model.predict_proba(X_test)[:,1]

In [11]:
pred = (probs >= 0.20).astype(int)

In [12]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test,pred)

print(cm)

[[32965 14743]
 [ 2751 59749]]


In [13]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

print("Accuracy", accuracy_score(y_test, pred))
print("Precision", precision_score(y_test, pred))
print("Recall", recall_score(y_test, pred))

print(classification_report(y_test,pred))

Accuracy 0.8412637921022067
Precision 0.802086130054234
Recall 0.955984
              precision    recall  f1-score   support

         0.0       0.92      0.69      0.79     47708
         1.0       0.80      0.96      0.87     62500

    accuracy                           0.84    110208
   macro avg       0.86      0.82      0.83    110208
weighted avg       0.85      0.84      0.84    110208



In [15]:
from IPython.display import display, Markdown

display(Markdown("""
## Threshold Selection

Threshold = 0.20 was selected based on business cost optimization.

Compared to a conservative threshold of 0.40, total cost decreased from 590,370 dollars to 348,815 dollars, resulting in a savings of 241,555 dollars.

This improvement comes from reducing missed churners (FN) from 5,560 to 2,751 (-50.5%). While false positives (FP) increased from 6,874 to 14,743 (+114.4%), this trade-off is acceptable because a missed churner costs 20x more than a false alarm.

### Cost Breakdown by Threshold

| Threshold | FN | FP | FN Cost (USD 100) | FP Cost (USD 5) | Total Cost |
|---:|---:|---:|---:|---:|---:|
| 0.40 | 5,560 | 6,874 | 556,000 | 34,370 | 590,370 |
| 0.35 | 4,786 | 8,289 | 478,600 | 41,940 | 520,540 |
| 0.30 | 4,099 | 9,973 | 409,900 | 49,865 | 459,765 |
| 0.25 | 3,416 | 12,043 | 341,600 | 60,215 | 401,815 |
| 0.20 | 2,751 | 14,743 | 275,100 | 73,715 | 348,815 |
| 0.15 | 2,081 | 18,473 | 208,100 | 92,365 | 300,465 |
| 0.10 | 1,322 | 23,544 | 132,200 | 117,720 | 249,920 |
| 0.05 | 567 | 31,765 | 56,700 | 158,825 | 215,525 |

Although 0.05 yields the lowest cost, the high volume of false positives (31,765) makes it operationally impractical. Therefore, 0.20 is selected as the optimal balance between cost reduction and actionable workload.
"""))


## Threshold Selection

Threshold = 0.20 was selected based on business cost optimization.

Compared to a conservative threshold of 0.40, total cost decreased from 590,370 dollars to 348,815 dollars, resulting in a savings of 241,555 dollars.

This improvement comes from reducing missed churners (FN) from 5,560 to 2,751 (-50.5%). While false positives (FP) increased from 6,874 to 14,743 (+114.4%), this trade-off is acceptable because a missed churner costs 20x more than a false alarm.

### Cost Breakdown by Threshold

| Threshold | FN | FP | FN Cost (USD 100) | FP Cost (USD 5) | Total Cost |
|---:|---:|---:|---:|---:|---:|
| 0.40 | 5,560 | 6,874 | 556,000 | 34,370 | 590,370 |
| 0.35 | 4,786 | 8,289 | 478,600 | 41,940 | 520,540 |
| 0.30 | 4,099 | 9,973 | 409,900 | 49,865 | 459,765 |
| 0.25 | 3,416 | 12,043 | 341,600 | 60,215 | 401,815 |
| 0.20 | 2,751 | 14,743 | 275,100 | 73,715 | 348,815 |
| 0.15 | 2,081 | 18,473 | 208,100 | 92,365 | 300,465 |
| 0.10 | 1,322 | 23,544 | 132,200 | 117,720 | 249,920 |
| 0.05 | 567 | 31,765 | 56,700 | 158,825 | 215,525 |

Although 0.05 yields the lowest cost, the high volume of false positives (31,765) makes it operationally impractical. Therefore, 0.20 is selected as the optimal balance between cost reduction and actionable workload.
